In [22]:
from pathlib import Path
import pandas as pd

# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

# Read proper tables
MatchEventInfo_df = pd.read_parquet(zip_folder / "event_all.parquet")

MatchHomeTeamInfo_df = pd.read_parquet(zip_folder / "home_team_all.parquet")

MatchAwayTeamInfo_df = pd.read_parquet(zip_folder / "away_team_all.parquet")

MatchTournamentInfo_df = pd.read_parquet(zip_folder / "tournament_all.parquet")

In [23]:
# From MatchEventInfo, select match_id and winner_code, then drop duplicates
MatchEventInfo_df = MatchEventInfo_df[["match_id","winner_code"]]
MatchEventInfo_df.drop_duplicates(subset="match_id", inplace=True)

# Select desired columns from MatchHomeTeamInfo and MatchHomeTeamInfo
MatchHomeTeamInfo_df = MatchHomeTeamInfo_df[["match_id","current_rank","date"]].copy()
MatchHomeTeamInfo_df.rename(columns={"current_rank":"home_rank"},inplace=True)

MatchAwayTeamInfo_df = MatchAwayTeamInfo_df[["match_id","current_rank","date"]].copy()
MatchAwayTeamInfo_df.rename(columns={"current_rank":"away_rank"},inplace=True)

# Merge 2 tables to get both teams' information in a row
result_df = MatchAwayTeamInfo_df.merge(MatchHomeTeamInfo_df, on=["match_id","date"])

# Merge the result table with MatchEventInfo to get the winner_code
result_df = result_df.merge(MatchEventInfo_df, on="match_id", how = "left")

# Drop date and null values on winner_code, because we do not need them!
result_df.drop(columns="date", inplace=True)
result_df.dropna(subset=["winner_code"], inplace=True)
result_df.drop_duplicates(inplace=True)

In [24]:
# Now, we have match_id, away_rank, how_rank and winner_code
# Let's find tournament name, too
MatchTournamentInfo_df = MatchTournamentInfo_df[["match_id","tournament_name"]]
MatchTournamentInfo_df.drop_duplicates(subset="match_id", inplace=True)

result_df = result_df.merge(MatchTournamentInfo_df, on="match_id", how = "left")

# Drop NaN values
result_df.dropna(subset="away_rank", inplace=True)
result_df.dropna(subset="home_rank", inplace=True)

In [25]:
# Now create wonder column!
result_df["wonder"] = (((result_df["winner_code"] == 1) & (result_df["home_rank"] > result_df["away_rank"])) |
                      ((result_df["winner_code"] == 2) & (result_df["away_rank"] > result_df["home_rank"])))

In [39]:
# Now, time to create wonder table and describe the RATE!
wonder_df = (result_df.groupby("tournament_name")
    .agg(total_matches=("match_id", "count"),
         wonder_matches=("wonder", "sum")).reset_index())

wonder_df["wonder_rate"] = (wonder_df["wonder_matches"] /wonder_df["total_matches"] * 100)

wonder_df = wonder_df.sort_values("wonder_rate",ascending=False)

wonder_df[wonder_df["total_matches"] > 20]


,tournament_name,total_matches,wonder_matches,wonder_rate
129,"Glasgow, Great Britain",40,27,67.500000
455,"Trnava, Singles Main, W-ITF-SVK-01A",32,20,62.500000
196,"Kofu, Singles Main, W-ITF-JPN-01A",30,18,60.000000
328,"Pretoria, Singles Main, W-ITF-RSA-01A",21,12,57.142857
456,"Trnava, Singles Main, W-ITF-SVK-02A",32,18,56.250000
...,...,...,...,...
80,"Burnie 2, Australia",37,5,13.513514
122,"Faro, Singles Main, M-ITF-POR-03A",25,3,12.000000
208,"Linz, Austria",21,2,9.523810
106,"Croissy-Beaubourg, Singles Q., W-ITF-FRA-10A",24,2,8.333333
